# White Castle rules assistant

An assistant that answers questions about the rules of the board game **White Castle**, in Spanish. It uses RAG (retrieval-augmented generation): instead of answering from memory, it first looks up the relevant passages of the official rulebook and then answers from them, citing the page.

**The starting point** is the Spanish rulebook: a 20-page scanned PDF where every page is an image, with no text to search. The notebook follows the path that turns it into an assistant:

| Section | Step | What happens |
|---|---|---|
| 1.1 | PDF → text | Gemini transcribes each page image to Markdown |
| 1.2 | Text → chunks | The text is cleaned and split into chunks that keep their page and section |
| 1.3 | Chunks → vectors | Gemini embeddings, stored in a local ChromaDB collection |
| 1.4 | Question → rules | Test questions check that the search finds the right rules |
| 2 | Rules → answer | LangGraph agent: retrieve the rules, then generate the answer with Gemini |
| 3 | Answer → conversation | Memory between turns, for follow-up questions |
| 4 | Conversation | Chat cell and example questions |

How to install and run it (dependencies, Gemini API key, where to place the rulebook): see the README.

## 0. Setup

Imports, the configuration shared by the whole notebook (paths and models) and the Gemini API key, read from `.env` and never printed.

The parameters of each step live in its own section, next to the explanation of their values.

In [1]:
# Imports
import base64
import os
import re
import statistics
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [2]:
# Paths
PROJECT_DIR = Path.cwd()
ENV_PATH = PROJECT_DIR / ".env"
PDF_PATH = PROJECT_DIR / "data" / "white_castle.pdf"
TRANSCRIPT_DIR = PROJECT_DIR / "data" / "transcript"
CHROMA_DIR = PROJECT_DIR / "chroma_db"

# Models
CHAT_MODEL = "gemini-3.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-2"

In [3]:
def load_api_key():
    """Gemini API key: .env → environment variable GOOGLE_API_KEY."""
    load_dotenv(ENV_PATH)
    if not os.getenv("GOOGLE_API_KEY"):
        raise ValueError(f"GOOGLE_API_KEY is empty or missing in {ENV_PATH}")


load_api_key()

## 1. Knowledge base

The agent can only answer as well as it searches. This part turns the scanned rulebook into a knowledge base the agent can search, and ends by checking that the search finds the right rules.

### 1.1 From a scanned PDF to text

The PDF has no text layer: each page is a single JPEG image. Instead of OCR, Gemini reads each image and transcribes it to Markdown:
- **One page per request:** with the whole PDF at once the model tends to summarise, and the page of each rule would be lost. If a page fails, only that page is repeated.
- **Icons that carry rules are described:** in a board game many rules (costs, rewards) only appear as icons, so they are described briefly between brackets, for example `[Icono: 2 monedas]`. Decorative illustrations are ignored.
- **Done once:** each page is saved to `data/transcript/page_NN.md` and skipped in later runs, so the quota is only spent once.

In [4]:
# Constants
TRANSCRIPTION_PROMPT = """You are transcribing one page of the Spanish rulebook of the board game White Castle.
- Transcribe all the text literally, in Spanish. Do not translate, summarize or correct it.
- Mark section titles with Markdown headings: # for main titles, ## for subsections.
- When an icon, diagram or illustrated example carries rules information, describe it briefly in Spanish between square brackets, for example: [Icono: 2 monedas].
- Ignore purely decorative illustrations, page numbers and repeated headers or footers.
- Return only the transcription, with no comments before or after it."""

In [5]:
def transcribe_manual():
    """Rulebook PDF → one Markdown file per page in TRANSCRIPT_DIR (pages already done are skipped)."""
    TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
    llm = ChatGoogleGenerativeAI(model=CHAT_MODEL)
    pages = PdfReader(PDF_PATH).pages
    transcribed = [transcribe_page(llm, page, number) for number, page in enumerate(pages, start=1)]
    print(f"{len(pages)} pages: {sum(transcribed)} transcribed now, {len(pages) - sum(transcribed)} already on disk")


def transcribe_page(llm, page, number):
    """One page: scanned image → Gemini transcription → page_NN.md (False if it was already on disk)."""
    output_path = TRANSCRIPT_DIR / f"page_{number:02d}.md"
    if output_path.exists():
        return False
    message = build_page_message(page)
    reply = llm.invoke([message])
    output_path.write_text(reply.text, encoding="utf-8")
    print(f"Page {number:02d}: transcribed ({len(reply.text)} characters)")
    return True


def build_page_message(page):
    """Message for Gemini: transcription prompt + the page JPEG image as base64."""
    image_b64 = base64.b64encode(page.images[0].data).decode()
    return HumanMessage(content=[
        {"type": "text", "text": TRANSCRIPTION_PROMPT},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_b64}"},
    ])


transcribe_manual()

20 pages: 0 transcribed now, 20 already on disk


### 1.2 Cleaning and chunking

The agent does not search whole pages but chunks: pieces small enough to hold one rule each. The transcription becomes chunks in four steps:
- **Cleaning:** removes the transcription noise that repeats (formula marks, arrow bullets, warning signs, decorative images). Isolated typos are left as they are: fixing them by hand does not pay off for the search.
- **Sections first:** the text is split by its Markdown headings, so a rule is not cut in two. A page that starts mid-section keeps the section of the previous page.
- **Then size:** sections longer than `CHUNK_SIZE` characters are split again, with some overlap so that a sentence on the border is not lost.
- **Each chunk carries its context:** its page and section are saved as metadata, to cite the source. The section path is also written at the start of the text, because the embedding only sees the text: a loose sentence like "pay 2 coins" means little without its section.

In [6]:
# Constants
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

HEADERS = [("#", "h1"), ("##", "h2"), ("###", "h3")]

# Cleaning rules: (regex pattern, replacement)
CLEANING_RULES = [
    (r"\$([^$\n]+)\$",                        r"\1"),  # formula marks: $10+10$ → 10+10
    (r"\[Icono: Flecha roja\]\s*",            "- "),   # arrow bullets → list items
    (r"\[Icono: Signo de exclamación\]\s*",   ""),     # warning sign before "Atención"
    (r"\*?\[Ilustración decorativa:[^\]]*\]\*?", ""),  # decorative images
    (r"\n{3,}",                               "\n\n"), # extra blank lines
]

In [7]:
def build_chunks():
    """Transcript pages → cleaned text → sections → chunks with page and section metadata."""
    pages = load_pages()
    sections = split_into_sections(pages)
    chunks = split_long_sections(sections)
    print_chunk_summary(chunks)
    return chunks


def load_pages():
    """Transcript files → list of (page number, cleaned text)."""
    pages = []
    for path in sorted(TRANSCRIPT_DIR.glob("page_*.md")):
        number = int(path.stem.split("_")[1])
        pages.append((number, clean_text(path.read_text(encoding="utf-8"))))
    return pages


def clean_text(text):
    """Raw transcription → text without the noise listed in CLEANING_RULES."""
    for pattern, replacement in CLEANING_RULES:
        text = re.sub(pattern, replacement, text)
    return text.strip()


def split_into_sections(pages):
    """
    Pages → one document per section:
        - Metadata: headings (h1, h2, h3) and page number
        - Text at the top of a page with no heading keeps the section of the previous page
    """
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=HEADERS)
    sections = []
    last_headers = {}
    for number, text in pages:
        for section in splitter.split_text(text):
            if section.metadata:
                last_headers = dict(section.metadata)
            else:
                section.metadata = dict(last_headers)
            section.metadata["section"] = " > ".join(section.metadata.values())
            section.metadata["page"] = number
            sections.append(section)
    return sections


def split_long_sections(sections):
    """Sections → chunks of at most CHUNK_SIZE characters, each starting with its section title."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = splitter.split_documents(sections)
    for chunk in chunks:
        chunk.page_content = f"{chunk.metadata['section']}\n\n{chunk.page_content}"
    return chunks


def print_chunk_summary(chunks):
    """Chunk count, length statistics and one example chunk."""
    lengths = [len(chunk.page_content) for chunk in chunks]
    print(f"Chunks: {len(chunks)}")
    print(f"Length (characters): min {min(lengths)}, median {statistics.median(lengths):.0f}, max {max(lengths)}")
    print(f"\nExample (chunk 10, page {chunks[10].metadata['page']}):\n{chunks[10].page_content}")


chunks = build_chunks()

Chunks: 76
Length (characters): min 118, median 616, max 1087

Example (chunk 10, page 2):
PREPARACIÓN DE LA PARTIDA > PUENTES

7 [Icono: 7 en cuadro rojo] Se colocan los puentes en sus espacios del río (si lo preferís, podéis jugar sin los puentes troquelados). Tomad 3/4/5 dados de cada color en función de si seis 2/3/4 jugadores. Devolved a la caja los dados que sobren.  
Tirad los dados de cada color y colocadlos en el puente que les corresponda, ordenándolos de manera ascendente según su valor de izquierda a derecha. Es decir, el valor más bajo a la izquierda del puente junto a la linterna, y el más alto a la derecha. Los demás se colocan en el centro del puente, en orden.


### 1.3 Vector store

Each chunk is turned into a vector with Gemini embeddings: chunks that talk about similar things get close vectors, so a question can find its rules by meaning and not only by exact words. The vectors are stored in a local ChromaDB collection (`chroma_db/`):
- If the collection already exists, it is loaded from disk and no embeddings are spent
- After changing the cleaning or chunking settings, set `REBUILD_VECTOR_STORE = True` once to rebuild it: the collection cannot tell by itself that the chunks have changed

In [8]:
# Constants
COLLECTION_NAME = "white_castle_rules"
REBUILD_VECTOR_STORE = False

In [9]:
def load_vector_store(chunks):
    """Chunks → ChromaDB collection in CHROMA_DIR (loaded from disk if it is already built)."""
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL),
        persist_directory=str(CHROMA_DIR),
    )
    stored = len(vector_store.get()["ids"])
    if REBUILD_VECTOR_STORE or stored == 0:
        index_chunks(vector_store, chunks)
    else:
        print(f"Vector store loaded from disk: {stored} chunks")
    return vector_store


def index_chunks(vector_store, chunks):
    """Empty the collection, then embed and store every chunk with a stable id."""
    vector_store.reset_collection()
    ids = [f"chunk_{number:03d}" for number in range(len(chunks))]
    vector_store.add_documents(chunks, ids=ids)
    print(f"Vector store built: {len(chunks)} chunks indexed")


vector_store = load_vector_store(chunks)

Vector store loaded from disk: 76 chunks


### 1.4 Does the search find the right rules?

Before building the agent on top of it, the search is checked with questions whose answer is known to be on a given page of the rulebook:
- **Seven direct questions**, one per key area of the rules
- **Three harder ones**, written as a player would ask them: each needs several rules at once, sometimes from different pages
- **A long version of the first hard one**, with a full game situation, the name of the game and a word that is not the rulebook's (*linterna* for *lámpara*)

For each question, the table shows the expected pages, the positions among the top `RETRIEVAL_K` chunks where a chunk from those pages appears, and the top chunk.

In [10]:
# Constants
RETRIEVAL_K = 5

LONG_DICE_QUESTION = (
    "En The White Castle, un jugador quiere coger un dado de valor 2 y colocarlo en un espacio cuyo valor de referencia es 5. "
    "Tiene 1 moneda, por lo que no puede pagar las 3 monedas de diferencia. Sin embargo, al coger ese dado "
    "activa su bonificación de linterna, que le proporcionaría monedas suficientes para cubrir la diferencia. "
    "¿Puede realizar la jugada? Explica el orden exacto de resolución, si puede utilizar la recompensa de la "
    "linterna para pagar el dado y qué alternativa tiene si no puede efectuar esa colocación."
)

# Test questions: (question, rulebook pages with the answer)
TEST_QUESTIONS = [
    ("¿Se pueden apilar dados en una partida a 2 jugadores?",                              (5,)),
    ("¿Qué pasa si coloco un dado de valor inferior al de la casilla?",                     (5,)),
    ("¿Cuántos cortesanos caben en la puerta del castillo?",                                (10,)),
    ("¿Cuál es el máximo de recursos de cada tipo que puedo tener?",                        (5,)),
    ("¿Cuánto cuesta subir un cortesano dos pisos?",                                        (10,)),
    ("¿Cómo puntúan los guerreros al final de la partida?",                                 (14,)),
    ("En el modo solitario, ¿el clan rival puntúa los recursos sobrantes?",                 (18,)),
    ("¿Puedes pagar un dado con los recursos que obtienes de la propia acción?",            (5, 8)),
    ("¿Puedes ascender dos veces o colocar dos cortesanos en una misma acción?",            (10,)),
    ("¿Puede un jardinero recién colocado activarse inmediatamente al final de una ronda?", (13, 10)),
    (LONG_DICE_QUESTION,                                                                    (5, 8)),
]

In [11]:
def check_retrieval():
    """Every test question → one table row, then how many questions found an expected page."""
    print(f"{'Question':<58} {'Expected':<9} {'Found at':<10} Top chunk")
    found = 0
    for question, expected_pages in TEST_QUESTIONS:
        chunks = vector_store.similarity_search(question, k=RETRIEVAL_K)
        ranks = [rank for rank, chunk in enumerate(chunks, start=1) if chunk.metadata["page"] in expected_pages]
        found += bool(ranks)
        print_row(question, expected_pages, ranks, chunks[0])
    print(f"\n{found}/{len(TEST_QUESTIONS)} questions with a chunk from an expected page in the top {RETRIEVAL_K}")


def print_row(question, expected_pages, ranks, top_chunk):
    """
    One aligned table row:
        Example: '¿Cuánto cuesta subir...'  p10  1, 3, 4  p10 CORTESANOS
        - Questions longer than the column are shortened
        - Icon descriptions are removed from the section title
    """
    short_question = question if len(question) <= 56 else question[:53] + "..."
    expected = ", ".join(f"p{page}" for page in expected_pages)
    found_at = ", ".join(str(rank) for rank in ranks) or "-"
    section = re.sub(r"\s*\[Icono:[^\]]*\]", "", top_chunk.metadata["section"])
    top = f"p{top_chunk.metadata['page']} {section[:45]}"
    print(f"{short_question:<58} {expected:<9} {found_at:<10} {top}")


check_retrieval()

Question                                                   Expected  Found at   Top chunk
¿Se pueden apilar dados en una partida a 2 jugadores?      p5        1, 3       p5 APILAR DADOS:
¿Qué pasa si coloco un dado de valor inferior al de l...   p5        1, 3       p5 USO DE LOS DADOS > PROCEDIMIENTO PARA USAR LO
¿Cuántos cortesanos caben en la puerta del castillo?       p10       1, 2, 3, 4 p10 CORTESANOS > A
¿Cuál es el máximo de recursos de cada tipo que puedo...   p5        1          p5 OBTENER Y PAGAR RECURSOS:
¿Cuánto cuesta subir un cortesano dos pisos?               p10       1, 3, 4    p10 CORTESANOS > B
¿Cómo puntúan los guerreros al final de la partida?        p14       1, 4       p14 FINAL DE PARTIDA > 3 Puntuación por los miemb
En el modo solitario, ¿el clan rival puntúa los recur...   p18       1          p18 Puntuación Final
¿Puedes pagar un dado con los recursos que obtienes d...   p5, p8    1, 2, 3, 4 p8 EL DOMINIO PERSONAL
¿Puedes ascender dos veces o colocar dos co

**What the check shows**
- **Direct questions:** the top chunk comes from the expected page in all seven.
- **Harder questions, short form:** the top chunk also comes from an expected page. When the answer is spread over two pages (paying for a die: pages 5 and 8; gardeners: pages 13 and 10), both pages appear in the top 5.
- **Long question:** the weakest case. The top chunk is *El castillo* (page 6), pulled by the words "The White Castle" of the question, and the expected chunks appear in positions 3 (paying the difference) and 5 (the order of the steps).
- **Why `RETRIEVAL_K = 5`:** with 3 chunks, the long question would lose the order of the steps, which is half of its answer. Five chunks of at most about 1,100 characters are still a small context for Gemini.
- **The search brings the pieces, not the answer:** in the harder questions the rulebook never states the answer literally (for example, "the lamp reward cannot pay for the die"). It has to be deduced from the order of the steps, which is the job of the agent and its system prompt (section 2).

## 2. RAG agent

*To be built.* A LangGraph graph with two steps: **retrieve** (the question → the top `RETRIEVAL_K` chunks of the rulebook) → **generate** (Gemini answers from those chunks only, following the system prompt). It includes the system prompt and the reasons behind each of its rules.

## 3. Conversation memory

*To be built.* The agent remembers the previous turns of the conversation, so it understands follow-up questions that lean on an earlier answer. It ends with an example that shows it.

## 4. Chat and examples

*To be built.* A chat cell to talk to the agent, and at least five example questions with their answers.